# 04 PyTorch Model Evaluation v1

PyTorch evaluation report parity for GreenSpace CNN. This notebook mirrors the TensorFlow NB04 outputs for loss monitoring, threshold tuning, and split-wise reports.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('MPLCONFIGDIR', '/private/tmp')

print('PROJECT_ROOT:', PROJECT_ROOT)

PROJECT_ROOT: /Users/starsrain/2025_codeProject/GreenSpace_CNN


In [2]:
import numpy as np
import pandas as pd
import torch

from src_torch.config import TORCH_DATA_CONFIG
from src_torch.data import load_split_dfs, resolve_split_schema
from src_torch.evaluation import (
    evaluate_all_splits,
    evaluate_loss_monitoring,
    find_latest_pytorch_checkpoint,
    infer_run_tag_and_variant,
    load_torch_checkpoint_model,
    predict_split,
    save_evaluation_outputs,
    tune_thresholds_f1,
)
from src_torch.training import resolve_device

split_dfs = load_split_dfs()
train_df = split_dfs['train']
val_df = split_dfs['val']
test_df = split_dfs['test']
schema = resolve_split_schema(train_df)
binary_cols = schema.binary_cols
bin_names = schema.bin_names

print('torch:', torch.__version__)
print('Loaded splits:', {k: len(v) for k, v in split_dfs.items()})
print('Binary labels:', binary_cols)

torch: 2.10.0
Loaded splits: {'train': 3067, 'val': 1022, 'test': 1022}
Binary labels: ['sports_field_p', 'multipurpose_open_area_p', 'children_s_playground_p', 'water_feature_p', 'walking_paths_p', 'built_structures_p', 'parking_lots_p']


## Load PyTorch Checkpoint

In [3]:
USE_MANUAL_MODEL = True
MANUAL_MODEL_PATH = PROJECT_ROOT / 'models/runs/PyTorch_20260531_211840/best_mcmae_PyTorch_20260531_211840.pt'
## /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840/best_mcmae_PyTorch_20260531_211840.pt
if USE_MANUAL_MODEL:
    model_path = Path(MANUAL_MODEL_PATH)
else:
    model_path = find_latest_pytorch_checkpoint(preferred_variant='best_mcmae')

assert model_path.exists(), f'Missing PyTorch checkpoint: {model_path}'
run_tag, EVAL_VARIANT = infer_run_tag_and_variant(model_path)
device = resolve_device('auto')
model, model_config, checkpoint = load_torch_checkpoint_model(model_path, device=device)

print('Loaded checkpoint:', model_path)
print('run_tag:', run_tag)
print('EVAL_VARIANT:', EVAL_VARIANT)
print('device:', device)

Loaded checkpoint: /Users/starsrain/2025_codeProject/GreenSpace_CNN/models/runs/PyTorch_20260531_211840/best_mcmae_PyTorch_20260531_211840.pt
run_tag: PyTorch_20260531_211840
EVAL_VARIANT: best_mcmae
device: mps


## Predict Splits

In [4]:
# Set to an integer for quick debug; use None for full split reports.
EVAL_MAX_BATCHES = None
EVAL_BATCH_SIZE = TORCH_DATA_CONFIG['batch_size']

predictions_by_split = {}
for split in ('train', 'val', 'test'):
    print(f'Predicting {split}...')
    predictions_by_split[split] = predict_split(
        model,
        split,
        device=device,
        batch_size=EVAL_BATCH_SIZE,
        max_batches=EVAL_MAX_BATCHES,
    )
    df_used, preds, loss_metrics = predictions_by_split[split]
    print(split, 'n=', len(df_used), 'loss=', round(loss_metrics['loss'], 4))

Predicting train...
train n= 3067 loss= 1.8337
Predicting val...
val n= 1022 loss= 2.3958
Predicting test...
test n= 1022 loss= 2.3532


## Loss Monitoring

In [5]:
loss_monitor_df = evaluate_loss_monitoring(predictions_by_split, binary_cols)
display(loss_monitor_df.round(4))

,split,loss,bin_head_loss,shade_head_loss,score_head_loss,veg_head_loss,bin_head_binary_accuracy,bin_head_pr_auc,shade_head_sparse_categorical_accuracy,score_head_mae,veg_head_mae
0,train,1.8337,0.3117,0.6216,0.4975,0.4029,0.8503,0.8203,0.7209,0.4975,0.4029
1,val,2.3958,0.3562,0.8724,0.6349,0.5322,0.8340,0.7749,0.6191,0.6349,0.5322
2,test,2.3532,0.3324,0.8817,0.6300,0.5091,0.8372,0.7970,0.5918,0.6300,0.5091


## Threshold Tuning

In [6]:
val_used_df, val_preds, _ = predictions_by_split['val']
hard_names_val = [name for name in bin_names if name in val_used_df.columns]
if hard_names_val:
    y_bin_val_true = val_used_df[hard_names_val].fillna(0).astype(int).values
    pred_bin_val_aligned = np.stack([val_preds['bin_head'][:, bin_names.index(name)] for name in hard_names_val], axis=1)
    label_names = hard_names_val
else:
    y_bin_val_true = (val_used_df[binary_cols].fillna(0.0).astype(np.float32).values >= 0.5).astype(int)
    pred_bin_val_aligned = val_preds['bin_head']
    label_names = bin_names

thresholds_df = tune_thresholds_f1(y_bin_val_true, pred_bin_val_aligned, label_names)
best_thresholds = {
    row['label']: float(row['best_threshold'])
    for _, row in thresholds_df.iterrows()
    if np.isfinite(row['best_threshold'])
}

print('Tuned thresholds:', len(best_thresholds), '/', len(label_names))
display(thresholds_df.sort_values('best_f1', ascending=False).reset_index(drop=True).round(4))

Tuned thresholds: 7 / 7


,label,best_threshold,best_f1,best_precision,best_recall,pos_rate,n_pos,n,note
0,multipurpose_open_area,0.2408,0.9362,0.9029,0.9721,0.8053,823,1022,
1,walking_paths,0.3084,0.9067,0.8637,0.9541,0.7671,784,1022,
2,built_structures,0.2952,0.7986,0.7691,0.8305,0.4041,413,1022,
3,sports_field,0.2633,0.7510,0.7727,0.7305,0.2505,256,1022,
4,parking_lots,0.2234,0.7357,0.7021,0.7726,0.2926,299,1022,
5,water_feature,0.3498,0.5540,0.5413,0.5673,0.2035,208,1022,
6,children_s_playground,0.2376,0.4715,0.3381,0.7787,0.1194,122,1022,


## Full Split Evaluation

In [7]:
overall_split_df, per_label_split_df = evaluate_all_splits(
    predictions_by_split,
    binary_cols,
    best_thresholds,
)

print('Overall metrics by split')
display(overall_split_df.round(4))
print('Per-label metrics by split')
display(per_label_split_df.round(4))

Overall metrics by split


,split,n_samples,n_labels,overall_F1@0.5,overall_ROC_AUC,overall_PR_AUC,overall_F1@tuned,shade_acc_overall,shade_acc_conditional,shade_acc_paths_present,score_acc,veg_acc,score_mae,veg_mae,score_ce,veg_ce,score_mae_mean,veg_mae_mean
0,train,3067,7,0.7424,0.9077,0.8203,0.7643,0.7209,0.8304,0.8248,0.5667,0.6788,0.5659,0.4291,NaN,NaN,0.4974,0.4029
1,val,1022,7,0.6981,0.8760,0.7749,0.7362,0.6194,0.6816,0.6735,0.4706,0.5479,0.6984,0.5667,NaN,NaN,0.6350,0.5326
2,test,1022,7,0.7167,0.8941,0.7970,0.7444,0.5930,0.6531,0.6485,0.4824,0.6037,0.6864,0.5250,NaN,NaN,0.6307,0.5093


Per-label metrics by split


,split,label,support_pos,support_neg,P@0.5,R@0.5,F1@0.5,ROC_AUC,PR_AUC,thr_val,P@thr,R@thr,F1@thr
0,train,multipurpose_open_area,2419,648,0.9334,0.9496,0.9414,0.9337,0.9779,0.2408,0.9120,0.9810,0.9452
1,train,walking_paths,2334,733,0.8942,0.9417,0.9174,0.9155,0.9686,0.3084,0.8763,0.9653,0.9187
2,train,built_structures,1294,1773,0.8636,0.7287,0.7904,0.9178,0.8905,0.2952,0.7793,0.8539,0.8149
3,train,sports_field,820,2247,0.8983,0.7000,0.7868,0.9515,0.9036,0.2633,0.8151,0.8012,0.8081
4,train,parking_lots,954,2113,0.8374,0.6153,0.7094,0.9174,0.8382,0.2234,0.7279,0.8354,0.7779
5,train,water_feature,615,2452,0.7063,0.4927,0.5805,0.8438,0.6728,0.3498,0.5780,0.6146,0.5957
6,train,children_s_playground,381,2686,0.4957,0.4488,0.4711,0.8740,0.4904,0.2376,0.3545,0.7927,0.4899
7,val,multipurpose_open_area,823,199,0.9240,0.9307,0.9274,0.9105,0.9747,0.2408,0.9029,0.9721,0.9362
8,val,walking_paths,784,238,0.8762,0.9209,0.8980,0.8829,0.9589,0.3084,0.8637,0.9541,0.9067
9,val,built_structures,413,609,0.8427,0.6877,0.7573,0.8987,0.8567,0.2952,0.7691,0.8305,0.7986


## Save Evaluation Artifacts

In [8]:
saved_paths = save_evaluation_outputs(
    run_tag=run_tag,
    variant=EVAL_VARIANT,
    loss_monitor_df=loss_monitor_df,
    thresholds_df=thresholds_df,
    overall_df=overall_split_df,
    per_label_df=per_label_split_df,
)

for name, path in saved_paths.items():
    print(name, '->', path)

loss_monitor -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/monitoring_output/runs/PyTorch_20260531_211840/loss_monitor_best_mcmae.csv
thresholds -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/monitoring_output/runs/PyTorch_20260531_211840/thresholds_best_mcmae.csv
overall -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/report_outputs/runs/PyTorch_20260531_211840/overall_metrics_by_split_best_mcmae.csv
per_label -> /Users/starsrain/2025_codeProject/GreenSpace_CNN/report_outputs/runs/PyTorch_20260531_211840/per_label_metrics_by_split_best_mcmae.csv
